# Window Functions in SQL


## Why Window Functions Matter

Window functions let you:
- Rank data
- Compare rows
- Calculate running totals

👉 Without losing original rows

This is heavily used in:
- Analytics dashboards
- Data pipelines
- Interviews

👉 This is where beginners struggle and pros shine


## Prerequisites

- PostgreSQL running ([`README.md`](README.md)).
- This notebook **drops** table `sales` if it exists so inserts stay deterministic.


## Sample data


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS sales CASCADE")

cur.execute(
    """
CREATE TABLE IF NOT EXISTS sales (
        id SERIAL PRIMARY KEY,
        user_id INT,
        amount INT
    )
"""
)

cur.execute(
    """
INSERT INTO sales (user_id, amount)
VALUES
(1, 100),
(1, 200),
(1, 150),
(2, 300),
(2, 100),
(3, 400)
"""
)

conn.commit()
print("sales table ready.")


## Problem without a window function

`GROUP BY` collapses rows—you only see one row per group.


In [ ]:
cur.execute(
    """
SELECT user_id, SUM(amount)
FROM sales
GROUP BY user_id
"""
)

print(cur.fetchall())

# 👉 You lost row-level detail.


## Window function (basic)

Keep each sale row **and** attach the per-user total with `OVER (PARTITION BY …)`.


In [ ]:
cur.execute(
    """
SELECT user_id, amount,
SUM(amount) OVER (PARTITION BY user_id) as total_per_user
FROM sales
"""
)

print(cur.fetchall())

# 👉 Each row + total per user


## Ranking


In [ ]:
cur.execute(
    """
SELECT user_id, amount,
RANK() OVER (PARTITION BY user_id ORDER BY amount DESC) as rank
FROM sales
"""
)

print(cur.fetchall())


## Row number


In [ ]:
cur.execute(
    """
SELECT user_id, amount,
ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY amount DESC) as row_num
FROM sales
"""
)

print(cur.fetchall())


## Running total


In [ ]:
cur.execute(
    """
SELECT user_id, amount,
SUM(amount) OVER (PARTITION BY user_id ORDER BY amount) as running_total
FROM sales
"""
)

print(cur.fetchall())


## Real interview query

**Top purchase per user** — filter the ranked subquery to `rn = 1`.


In [ ]:
cur.execute(
    """
SELECT *
FROM (
    SELECT user_id, amount,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY amount DESC) as rn
    FROM sales
) t
WHERE rn = 1
"""
)

print(cur.fetchall())

# 👉 "Top purchase per user" — classic pattern


## Practice

1. Find 2nd highest purchase per user
2. Calculate running total across all users
3. Rank users by total spending


## Assignment

Using the **`orders`** table (from [`02_joins_and_aggregations.ipynb`](02_joins_and_aggregations.ipynb) — recreate it if needed):

1. Find:
   - Top order per user
   - Running total per user
   - Rank users by total spending

**Bonus:** combine joins + window functions (e.g. user names from `users`).


## Interview Questions

1. What is a window function?
2. Difference between RANK and ROW_NUMBER?
3. When do you use window functions vs GROUP BY?
4. Explain PARTITION BY


## What you just learned

- **Advanced SQL** — row-level + analytic logic together
- **Ranking** and **running** metrics

👉 Pattern strong candidates use in interviews and production SQL.


---

**Next:** **Schema design** — model tables, normalization, avoid brittle schemas.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Run **Sample data** → **Real interview query** in order (fresh kernel).
- [ ] **Practice:** 2nd highest per user (`ROW_NUMBER` / `RANK` + filter `rn = 2`); running total **global** (`ORDER BY` in window without partition, or single partition); rank **users** by sum (CTE + window).
- [ ] **Assignment:** rebuild `orders` + `users` if needed, then windows + optional **JOIN** to names.
- [ ] Answer interview questions out loud; write one sentence on **frames** vs **PARTITION BY**.
